# 03 · Indexing and broadcasting real data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb)

*Part III · exercise · 15 min*

> 🇪🇸 **Indexación y broadcasting con datos reales** — Seleccionar medidas reales por nombre, comparar subconjuntos con máscaras booleanas y detectar píxeles de varianza cero.

Select named measurements from real tumour-sample data, compare meaningful subsets, then standardize real image data safely.

## What you will be able to do

- Select a named column of real data by name, never by a hard-coded number.
- Use fancy indexing and boolean masks to select meaningful subsets of rows.
- Standardize a real data matrix with broadcasting.
- Recognise zero-variance pixels and explain why real image datasets can contain them.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_digits

bc = load_breast_cancer()
X, y = bc.data, bc.target          # (569, 30); y: 0 = malignant, 1 = benign
names = list(bc.feature_names)
print(X.shape, len(names))

## Why this matters

> 🇪🇸 Elegir la columna equivocada no da error: devuelve otra medida real, y el análisis puede continuar con una respuesta convincente pero equivocada.

The `breast_cancer` dataset contains 569 real breast-mass samples with 30 numerical features computed from digitized images of fine-needle aspirates. Selecting the wrong column does not necessarily produce an error — it can return a *different real measurement* while the rest of the analysis keeps running.

In research, that makes results harder to reproduce and easier to misinterpret. In any decision-support pipeline, selecting the wrong feature can quietly change the conclusion.

**In tech**, the identical indexing operation appears on a `(users, items)` matrix when selecting one user's history before making a recommendation.

## Exercise 1 — indexing by name

> 🇪🇸 Indexación por nombre, nunca por número fijo.

In [ ]:
# TODO 1: Print X.shape. Say out loud what each axis means.

# TODO 2: Extract the column "mean radius" for all samples -> shape (569,).
#         Find its position with names.index(...). Do not hard-code a number.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(X.shape)                 # (569, 30)  patients x measurements

i = names.index("mean radius")
radius = X[:, i]               # book notation A_{:,j}
print(i, radius.shape)         # 0 (569,)

# names.index() rather than 0 because the column order is not yours to assume.
# If the dataset is ever reordered, the hard-coded version keeps running and
# keeps being wrong.

## Exercise 2 — fancy and boolean indexing

> 🇪🇸 Indexación avanzada y booleana.

In [ ]:
# TODO 3: Find the 5 samples with the LARGEST mean radius, then extract their
#         full 30-measurement profiles as one (5, 30) array, in ONE operation.
#
# TODO 4: Using boolean indexing, compare mean radius for malignant (y == 0)
#         against benign (y == 1) samples.
#         Describe the difference in THIS dataset; do not treat one feature
#         as a diagnostic rule.


In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
top5 = np.argsort(radius)[-5:]
profiles = X[top5, :]                               # (5, 30)
print(profiles.shape)

malignant_mean = radius[y == 0].mean()
benign_mean = radius[y == 1].mean()
print(malignant_mean, benign_mean)                  # about 17.5 vs 12.1

# Dataset-specific result:
# malignant samples have a larger mean radius ON AVERAGE in this dataset.
# That is a descriptive comparison, not a one-feature diagnostic rule.
#
# Purely synthetic random data would not preserve this real dataset
# relationship unless we explicitly designed it to do so.


`radius` was one column out of 30, picked because it happens to separate the
two groups well. Drag the slider below to look at all 30 — most separate far
less cleanly.

> 🇪🇸 Mueve el deslizador para ver las 30 medidas, una por una. La mayoría
> separa malignos de benignos mucho peor que el radio.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt

def show_feature(i):
    plt.close('all')
    col = X[:, i]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(col[y == 0], bins=30, alpha=0.6, label='malignant', color='#C44E52')
    ax.hist(col[y == 1], bins=30, alpha=0.6, label='benign', color='#4C72B0')
    ax.set_title(names[i])
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"malignant mean: {col[y == 0].mean():.3f}   "
          f"benign mean: {col[y == 1].mean():.3f}")

widgets.interact(show_feature,
                  i=widgets.IntSlider(min=0, max=len(names) - 1, step=1, value=0,
                                       description='feature'));

## Broadcasting, on real images

> 🇪🇸 **Broadcasting sobre imágenes reales.** Una operación muy común es estandarizar cada característica restando su media y dividiendo por su desviación estándar.

Broadcasting stretches a smaller array across a larger one without manually copying it. A common preprocessing operation is to standardize a data matrix — subtract the mean of each column, then divide by its standard deviation.

Run TODO 6 and **look at the result before continuing**. Something is wrong with it, and finding out what is the point of this block.

In [ ]:
images = load_digits().images          # (1797, 8, 8)
D = images.reshape(len(images), -1)    # (1797, 64)
print(D.shape)

## Exercise 3 — standardize, then find the trap

> 🇪🇸 Estandariza y encuentra el problema.

In [ ]:
# TODO 5: Compute the mean and std of each of the 64 pixels across all images.

# TODO 6: Standardize with broadcasting: (D - mean) / std.
#         RUN IT AND LOOK AT THE RESULT before continuing.

# TODO 7: You will find NaN. How many pixels have std == 0, and why would a real
#         handwritten digit image contain such pixels? Fix it, then verify no NaN.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
mean, std = D.mean(axis=0), D.std(axis=0)
print(mean.shape, std.shape)                        # (64,) (64,)

Z_bad = (D - mean) / std
print(np.isnan(Z_bad).any())                        # True

print((std == 0).sum())                             # 3
Z = (D - mean) / np.where(std == 0, 1.0, std)
print(np.isnan(Z).any())                            # False

# THREE PIXELS ARE ALWAYS DARK across all 1797 digit images.
# They are background/edge locations that are never activated in this dataset.
# Their standard deviation is exactly zero, so dividing by it produces NaN.
# np.where leaves those columns as centred zeros, which is appropriate for
# features that carry no variation in this dataset.

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(D.mean(axis=0).reshape(8, 8), cmap='gray')
zero_rows, zero_cols = np.where((std == 0).reshape(8, 8))
ax.scatter(zero_cols, zero_rows, s=250, marker='s',
           facecolors='none', edgecolors='#C44E52', linewidths=2)
ax.set_title('zero-variance pixels, marked')
ax.axis('off')
plt.show()


## What just happened

Two concrete patterns came directly from real datasets:

1. **In this breast-cancer dataset**, malignant samples have a larger mean radius on average — about 17.5 versus 12.1 for benign samples.
2. **Three pixel positions have zero variance** across all 1797 `load_digits` images. Dividing by their standard deviation therefore produces `NaN`.

The second result is the broadcasting trap to remember. A zero-variance feature is not necessarily a bug in your code; it can be a property of the data. You must detect it and handle it explicitly rather than silently propagating invalid values into later computations.

> 🇪🇸 **Qué ocurrió:** observaste dos patrones que provienen directamente de datos reales. En este conjunto de cáncer de mama, las muestras malignas tienen un radio medio mayor en promedio. En `load_digits`, tres posiciones de píxel tienen varianza cero en las 1797 imágenes, por lo que dividir por su desviación estándar produce `NaN`. La lección es detectar y manejar explícitamente las características sin variación antes de continuar con el análisis.

---

## Done with this section

Next up: **04 · Reshape and transpose** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb).

[← Back to the workshop site](../index.html) · [All notebooks](../notebooks.html) · [Handbook](../tensors_workshop_plan_with_quizzes.html)
